# darts_forecast explorer: which bar size, input length and prediction length predict the price path best?

**In one sentence.** For a small, diverse set of stocks, a Darts model reads the last *L* bars and predicts
the whole path of the next *H* bars; an optimizer chooses the **bar size** (1 to 60 minutes), the **input
length** (50 to 1,000 bars) and the **prediction length** (10 to 100 bars) for which the predicted path
agrees best with the actual one, point by point, averaged over all the stocks.

It is a **workbench for setting a baseline**: one model, one set of inputs, three knobs searched. It does
not trade and says nothing about profit.

**Where each part of the request is answered**

| you asked for | where |
|---|---|
| the bar size as a hyperparameter: 1, 2, 5, 10, 30 or 60 minutes | §0, §7 |
| input length 50 → 1,000 bars, prediction length 10 → 100 bars | §0, §7 (coarse steps; impossible combinations skipped) |
| optimize the agreement of the whole predicted path, MSE-like | §5 (the grade: the path's R² against "no change"), §7 |
| all stocks with the same settings in one run, the score averaged over them | §7 (one model per configuration for all 12; the grade is each stock's, averaged equally) |
| how the input and output lengths change the picture | §8 (predicted paths under several settings, accuracy by lead time, each knob's effect, maps, importance) |
| challenge the approach | §0 ("what can go wrong", each point measured on these stocks) |
| select the Darts model; a small, diverse list of stocks; no leakage; a smoke switch | §0 (`MODEL`, `SMOKE`), §1, §4, §6 |

**Roadmap**

| § | question |
|---|---|
| 0 | which settings, what can go wrong, and what do the words mean? |
| 1 | which stocks, and why these? |
| 2 | is their data complete, what did they do, and what do 1-minute bars hide? |
| 3 | what exactly is predicted, and from what? |
| 4 | which months train, which are graded, and how do we know nothing leaks? |
| 5 | one configuration end to end: predicted paths against actual paths |
| 6 | which Darts model? |
| 7 | the search: bar size × input length × prediction length |
| 8 | how do the three knobs change the picture? |
| 9 | does the winner hold on a month the optimizer never saw? |
| 10 | the checks, the summary, and what to try next |

**Run time.** `SMOKE = True` runs every step small in about a minute. The full run takes about 30–45 minutes
the first time (the search is most of it: 60 configurations, from seconds to 3 minutes each); every result
is saved in `cache/`, so a re-run takes seconds.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel, or run  uv sync --all-extras
import importlib.util
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger("darts").setLevel(logging.ERROR)  # Darts' notes about optional models it lacks

import pandas as pd

from trading_simple import strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine


def helpers(name: str):
    "One of the strategy folder's helper modules, loaded by path (strategy.py imports none)."
    path = strategies.STRATEGIES_DIR / "darts_forecast" / f"{name}.py"
    spec = importlib.util.spec_from_file_location(f"darts_{name}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


ex, charts = helpers("explorer"), helpers("charts")
rs = ex.rs  # research.py, as loaded by the explorer
settings = load_settings()
engine = make_engine(settings)
pd.set_option("display.max_colwidth", 90)

## 0 · The settings

**Question: which settings, what can go wrong, and what do the words mean?**

Every setting is in the next cell. `SMOKE = False` (the default) is the real run. Switch on `SMOKE = True`
to test every step end to end in about a minute: 4 stocks, 3 months, two bar sizes, a 2 × 2 × 2 search,
one training pass. Use it after any code change.

In [ ]:
SMOKE = True  # True: every step, small and fast (a test); False: the real run
MODEL = "tide"  # the Darts model of the search: tide | tsmixer | dlinear (§6 compares them)
WINDOW = ("2025-08-01", "2026-01-31")  # the months studied (inside TUNE, see below)
TRAIN_MONTHS, TEST_MONTHS = 2, 1  # the first fit learns from 2 months; the last month is TEST
GRANULARITIES = list(ex.GRANULARITIES)  # bar sizes the optimizer may pick (minutes)
INPUT_LENGTHS = list(ex.INPUT_LENGTHS)  # bars the model reads
PRED_LENGTHS = list(ex.PRED_LENGTHS)  # bars it predicts
N_TRIALS = 60  # configurations the optimizer measures
EPOCHS = 10  # at most this many training passes (stopped early on a held-out week)
DEFAULT = (5, 200, 20)  # (bar minutes, input, prediction) of the worked example (§5) and §6
COMPARE = ["tide", "tsmixer", "dlinear"]  # the models of §6
DRAWS = 200  # make-believe futures behind every luck band
SYMBOLS = None  # None: the picks of §1; or your own list, e.g. ["AAPL", "KO", "SMCI"]
READ_LOCKED = False  # True allows a WINDOW after the TUNE months (see below)

K = ex.knobs(smoke=SMOKE, model=MODEL, window=WINDOW, train_months=TRAIN_MONTHS,
             test_months=TEST_MONTHS, granularities=GRANULARITIES, inputs=INPUT_LENGTHS,
             outputs=PRED_LENGTHS, n_trials=N_TRIALS, epochs=EPOCHS, draws=DRAWS,
             compare=COMPARE, default=DEFAULT)
CHECKS = []  # every section adds its checks; §10 shows them all
display(ex.knob_table(K))

**Why these settings**

| setting | value | why |
|---|---|---|
| `GRANULARITIES` | 1, 2, 5, 10, 30, 60 | your list. Every bar is built from the stored 1-minute bars |
| `INPUT_LENGTHS` | 50, 100, 200, 400, 1000 | your range, on steps that about double: the optimizer cannot spend trials on small changes |
| `PRED_LENGTHS` | 10, 20, 50, 100 | the same for the prediction length |
| `MODEL` | `tide` | TiDE predicts the whole path at once and reads the inputs and the calendar; fast enough for long inputs. §6 compares TSMixer and DLinear on one configuration |
| `N_TRIALS` | 60 | about 60% of the configurations the data supports (§7); about 30–45 minutes, inside the 1–3 hour budget |
| `EPOCHS` | 10 | at most: training stops when the error on a held-out week stops improving for 2 passes, and keeps the best pass |
| `WINDOW` | Aug 2025 → Jan 2026 | six months inside the darts study's TUNE months: the locked TEST (Feb–Apr 2026) and HOLDOUT (May 2026 on) stay unread. `READ_LOCKED = True` allows a later window |
| `DEFAULT` | 5-min bars, input 200, predict 20 | the worked example: "from the last 2.6 sessions of 5-minute bars, predict the next 100 minutes, bar by bar" |

**What can go wrong** (each point measured on these stocks; the notebook guards against each)

| trap | what it does | the guard |
|---|---|---|
| bars mean different time spans | 1,000 bars are 2.6 sessions at 1 minute but 143 at 60 minutes, more than the whole window | combinations the data cannot support are skipped (§7) |
| some combinations are graded on very few outcomes | a 100-bar prediction at 60 minutes spans 14 sessions: 3 months hold ~4 independent ones per stock | each configuration needs ≥ 200 training windows and ≥ 10 independent validation paths per stock, and a held-out stretch of ≥ 5 independent paths |
| raw MSE always prefers short, fine paths | a path's variance grows with its length | the grade is relative: R² against "no change", averaged over lead times (§5) |
| "no change" is hard to beat | for stock returns the best squared-error forecast is close to 0; a network's paths are louder than what it knows | each configuration's paths are scaled by how much of them proved right on held-out data before the graded months (§5) |
| 1-minute bars hide bounce | a 1-minute move tends to reverse (bid-ask bounce, minutes without a trade): "predictable" but not tradable | a one-number AR(1) baseline is graded next to every configuration (§2, §5) |
| the best of many tries flatters itself | with 60 tries, a few look good by luck | a luck band for direction, the bar for the best of all tries, and a TEST month (§7–§9) |

**Words you will meet**

| word | meaning |
|---|---|
| bar, granularity | *g* minutes of trading built from the 1-minute bars; a session (09:30–16:00 New York) has 390 / *g* |
| input length *L* | how many past bars the model reads |
| prediction length *H* | how many future bars it predicts, all at once: the predicted **path** |
| decision | the moment a path is predicted: 10:30, 11:30, …, 15:30 (closes of every bar size) |
| bar move, typical move | a bar's close-to-close change, divided by that clock time's usual size over the last 20 sessions, so a calm and a wild stock, and the open and midday, are comparable |
| lead time | how far ahead a point of the path is: 1 … *H* bars, *g* … *g·H* minutes |
| R² against "no change" | 1 − (squared error of the predicted path) / (squared error of predicting no change), per stock and lead time: 0 = as good as saying nothing, above 0 = better. **The grade**, averaged over lead times and stocks |
| scale (β) | how much of a network's path survives held-out data: 0 = none (the path becomes "no change"), 1 = all |
| direction (corr) | the correlation of predicted and actual change, per stock and lead time, averaged: does the path point the right way? The scale cannot move it |
| luck band, z | where the direction lands when the outcomes have no direction; z = how many luck-SDs above luck's centre |
| AR(1) baseline | "the next bar repeats ρ × the last one", one number ρ per stock, fit on TRAIN |

## 1 · The stocks: small, diverse, with complete 1-minute data

**Question: which stocks, and why these?**

**The data rules.** A stock is a candidate only if it passes all five, in this order:

1. **1-minute data on every session of the window**: a missing day would silently stretch a path.
2. **A trade in at least 90% of the window's minutes**: a minute without a trade is stored as a copy of
   the last price; too many of them and the bars are stale.
3. **1-minute data over the last 6 months** (at least 95% of those sessions): the stock is still stored
   today. This counts bars only, it reads no prices.
4. **Listed on a US exchange, with a market cap** (today's, from the Nasdaq screener; ADRs count).
5. **A price of at least $5, and at least 40 daily returns before the window** (for its speed, below).

**The groups.** *Market cap* (today's): large ≥ $10 B, mid $2–10 B, small < $2 B. *Speed*: the stock's
daily volatility in the 3 months **before** the window, as a percentile inside its cap group: **slow** =
nearest the 10th percentile, **fast** = nearest the 90th. Two stocks per group and speed, preferring a
sector not picked yet: 12 stocks.

**What the choice may and may not see.** The data rules and today's market cap look at the window and
beyond, at data quality and size, never at a return of the window. One mild bias remains: a stock must
still be listed and stored today, which favours stocks that did not collapse after the window started.

In [ ]:
cands = ex.candidates(engine, WINDOW)  # every stock with 1-minute data in the window
print(f"window {WINDOW[0]} → {WINDOW[1]}: {cands.attrs['sessions']} sessions; "
      f"'last 6 months' = {cands.attrs['recent_from']} → {cands.attrs['recent_to']}; "
      f"market caps: {cands.attrs['listing']}")
display(ex.rule_funnel(cands))
picks = ex.pick(cands)  # two per cap group and speed, nearest the target percentile, sectors spread
display(ex.pick_table(picks))

**The figure.** Every stock that passes the rules (grey) by today's market cap and its speed before the
window, both on log scales; the picks coloured (circles slow, diamonds fast).

In [ ]:
charts.universe_map(cands, picks).show()
NAMES = SYMBOLS or ex.default_names(picks, smoke=K["smoke"])  # SMOKE: one of each extreme group
print(f"{len(NAMES)} stocks: {', '.join(NAMES)}")
CHECKS += [("every pick passes the five data rules", bool(cands.loc[picks["symbol"], "ok"].all()),
            f"{len(picks)} picks")]

## 2 · The data: complete, what the stocks did, and what 1-minute bars hide

**Question: is the data complete, and how do the bar sizes differ before any model?**

The panel holds every regular-session minute of the window plus the 21 sessions before it that the inputs
need for their history (the "warm-up"; no model learns from them). Prices are adjusted for dividends.

In [ ]:
panel, SP = ex.load_window(settings, engine, NAMES, K["window"], train_months=K["train_months"],
                           test_months=K["test_months"], read_locked=READ_LOCKED)
dq = ex.data_quality(panel, picks)
print(f"{len(panel.symbols)} stocks, {len(SP['sessions'])} sessions "
      f"{SP['sessions'][0]} → {SP['sessions'][-1]}, {len(panel.close):,} one-minute rows")
display(dq.round(1))
CHECKS += [
    ("every stock has every session", bool((dq["sessions"] == len(SP["sessions"])).all()),
     f"{len(SP['sessions'])} sessions"),
    ("every stock trades in >= 90% of its minutes",
     bool((dq["minutes with a trade %"] >= 90).all()),
     f"lowest {dq['minutes with a trade %'].min():.1f}%"),
    ("no day move above 60% (an unadjusted split would show as one)",
     bool((dq["largest day move %"] < 60).all()), f"largest {dq['largest day move %'].max():.1f}%"),
    ("the window stays inside TUNE (the locked months unread)",
     SP["window"][1] <= rs.WINDOWS["tune"][1] or READ_LOCKED,
     f"window ends {SP['window'][1]}; TUNE ends {rs.WINDOWS['tune'][1]}"),
]
charts.price_paths(panel, picks, SP).show()

**What 1-minute bars hide.** Before any model: how much does a bar's move tend to repeat (positive) or
reverse (negative) the previous bar's, inside one session? On 1-minute bars a move tends to reverse: the
bid-ask bounce (trades alternate between the bid and the ask) and the minutes without a trade (stored as a
copy of the last price, then a catch-up). A model can "predict" that, but it is not tradable: it lives
inside the spread. The last column is the R² that pattern alone would give at the next bar: the AR(1)
baseline of §5 measures it on every configuration.

In [ ]:
micro = ex.microstructure(panel, K["granularities"], before=SP["test"][0])  # TEST stays unread
display(micro)
print("minutes without a trade (%):", micro.attrs["no-trade minutes %"])

## 3 · What is predicted, and from what

**Question: what exactly does the model predict, and what does it read?**

**The target: the path.** Every bar's move from the previous bar's close (a session's first bar holds the
night's gap), in typical moves: divided by the usual size of that clock time's move over the previous 20
sessions (the open's bar by the open's usual size, gap included). A move beyond 10 typical moves counts as
10: one earnings jump would otherwise decide the grade alone. At a decision, the model reads the last *L*
bars and predicts the next *H* bar moves at once; added up they are the predicted path, and with the
decision's price, the predicted **price path**.

**The decisions.** Every configuration is graded at the same clock times: 10:30, 11:30, 12:30, 13:30, 14:30
and 15:30, closes of every bar size, so a 1-minute and a 60-minute configuration face the same market
moments. A path counts only if it ends inside the graded months and crosses no half-day afternoon, so a
longer path has somewhat fewer graded decisions (about 13% fewer at 60-minute bars, predicting 20).

**The inputs** (all known at the decision; one model learns every stock, with the stock as a static input):

In [ ]:
g0, L0, H0 = K["default"]
pin0 = ex.path_inputs(panel, g0)  # the worked example's grid: every input and the target
print(f"{g0}-minute bars: {len(pin0['g']['code']):,} bars per stock; the worked example reads {L0} "
      f"and predicts {H0}")
display(ex.feature_table(pin0, L0))

## 4 · The split, and why nothing leaks

**Question: which months train, which are graded, and how do we know no future data leaks into a
prediction?**

| part | what happens |
|---|---|
| **warm-up** (21 sessions) | inputs only: the 20-day inputs and the typical-move scale need history |
| **TRAIN** (2 months) | each configuration fits ONE model on these outcomes; its last 5 sessions (more when a long path needs more: at least 5 independent paths) are held out to stop training early and to measure the scale β |
| **VALIDATION** (3 months) | the paths are graded. **The optimizer chooses on these grades** |
| **TEST** (1 month) | graded **once**, at the end, for the chosen configurations only, each refit on TRAIN + VALIDATION (§9) |

**The boundaries, and the guard that can fail** (their results join the checks):

1. **The boundaries, as built**: training labels end before the held-out sessions, those end before the
   first graded decision, and every graded path starts at a decision time and ends inside its part (so a
   VALIDATION grade never reads a TEST price). The code enforces these; the checks document them.
2. **The truncation test** catches what the code does not intend: every 1-minute bar after a cut inside
   VALIDATION is replaced by random bars and the whole configuration re-run (inputs, fit, scale,
   predictions). Every path the network predicted before the cut, and the scale, must be identical to the
   last digit, and the later paths must change (else the test could not fail). It earned its place: it
   caught a scale computed over the whole panel, now fixed.

In [ ]:
display(ex.split_table(SP))
charts.explorer_split(SP, [(SP["validation"][0], "fit (TRAIN)"),
                           (SP["test"][0], "refit (TRAIN + VALIDATION)")]).show()

## 5 · One configuration end to end: predicted paths against actual paths

**Question: for one configuration, what does the model predict, and how close does the path come?**

The worked example (`DEFAULT`): 5-minute bars, input 200 bars (2.6 sessions), predict 20 bars (100
minutes). One TiDE model is fit for all the stocks on TRAIN, its paths are scaled by β (below), and every
decision in VALIDATION gets a predicted path.

In [ ]:
r0 = ex.run_path(panel, SP, K["default"], kind=K["model"], epochs=K["epochs"], draws=K["draws"])
CHECKS += ex.path_checks(r0, pin0, SP)
CHECKS += ex.path_truncation(panel, SP, K["default"], kind=K["model"], epochs=K["epochs"])
display(rs.checks_table(CHECKS[-5:]))
print(r0["fit"])

**The figure.** One stock at one decision: the actual price (black, 1-minute closes) and the model's
predicted price path from the decision (solid: the graded, scaled path; dotted: the network's own, before
scaling), converted to dollars with each bar's typical size as known at the decision.

In [ ]:
NAME, TS = ex.example(picks, NAMES, SP)  # a fast large cap, 12:30 on the 5th VALIDATION day
charts.path_picture(panel, ex.example_paths(panel, [(f"{g0}-min, input {L0}, predict {H0}", r0)],
                                            NAME, TS), NAME, TS).show()

**The grade.** For every stock and every lead time *h* (1 … *H* bars ahead), compare the predicted path
with the actual one over all decisions: **R² = 1 − Σ (predicted − actual)² / Σ actual²**. Predicting "no
change" (a flat path) scores exactly 0; a path that helps scores above 0; one that hurts, below. The grade
of a configuration is the mean over lead times, then over stocks (equal weights, so no stock dominates).

**Why the paths are scaled first (β).** For stock returns, the best squared-error forecast is close to "no
change": most of every move is unpredictable. A network trained on that noise still draws paths that are
too loud for what it knows, and loudness alone costs R². So before grading, each configuration's paths are
multiplied by β (0 … 1): the factor that makes them agree best, the way they are graded (every lead time
weighted equally), with the held-out sessions at the end of TRAIN (never the graded months). β = 0 means
"the held-out data found nothing in these paths": the configuration then predicts "no change" and scores 0.
So R² measures information that carries over, not how loudly a model guesses. The unscaled paths are
graded too ("R² unscaled").

**Direction and luck.** **Direction** is the correlation of predicted and actual change, per stock and lead
time, averaged: do the paths point the right way? Scaling cannot move it. Its **luck band** grades the same
paths against 200 make-believe futures in which every bar keeps its size but gets a random sign (one coin
per bar for all stocks): direction is gone, everything else stays. **z** is how many luck-SDs the
direction sits above luck's centre.

In [ ]:
row0 = pd.Series(ex._row(r0)).drop(["r2_h", "corr_h"])
display(row0.to_frame("worked example").T.round(4))
by_stock = r0["score"]["r2_by_name"].rename("R² against no change (scaled paths)")
display(by_stock.to_frame().T.round(4))

**The figures: agreement by lead time.** R² against "no change" at every point of the path, from 1 bar to
*H* bars ahead, then the direction the same way. The AR(1) baseline's grade is in the table above
(`ar1_r2`).

In [ ]:
charts.lead_curves(pd.DataFrame([ex._row(r0)]),
                   title="The worked example: R² by lead time (VALIDATION)").show()
charts.lead_curves(pd.DataFrame([ex._row(r0)]), metric="corr_h",
                   title="The worked example: direction by lead time (VALIDATION)").show()

## 6 · Which Darts model?

**Question: on the same stocks, inputs and configuration, how do the multi-step Darts models compare?**

In [ ]:
display(ex.models_table(K["compare"]))
comp = ex.compare_paths(panel, SP, K["compare"], K["default"], epochs=K["epochs"], draws=K["draws"])
display(comp[["r2", "beta", "r2_raw", "corr", "z", "share_pos", "ar1_r2", "epochs", "fit_s"]]
        .round(4))
charts.dots(ex.as_dots(comp.reset_index(), "model"), "corr", scale=1.0, unit="",
            title="Direction by model against luck (VALIDATION)").show()

**Look for:** a model whose dot sits outside its grey luck band (its paths point the right way more than
luck), and a scaled R² (`r2`) above 0. Differences inside the bands are not evidence: choose `MODEL` in §0
for what you want to learn.

## 7 · The search: bar size × input length × prediction length

**Question: which configuration's predicted paths agree best with the actual ones on VALIDATION?**

**The search space.** Three ladders, every step about double the last, nothing in between:

| knob | steps |
|---|---|
| bar size (minutes) | 1 · 2 · 5 · 10 · 30 · 60 |
| input length (bars) | 50 · 100 · 200 · 400 · 1,000 |
| prediction length (bars) | 10 · 20 · 50 · 100 |

**What the data supports.** A configuration is tried only with at least 200 training windows per stock
before its held-out sessions (at least 5 sessions, more when 5 independent paths need more) and at least
10 independent VALIDATION paths per stock (VALIDATION's bars divided by the prediction length). The table
shows which exist; long inputs and long predictions drop out at coarse bar sizes, where two months of
training data are too few bars.

In [ ]:
feas = ex.feasibility_table(SP, K["granularities"], K["inputs"], K["outputs"],
                            min_windows=K["min_windows"], min_outcomes=K["min_outcomes"])
display(feas)
n_ok = int((feas.filter(like="predict") == "ok").sum().sum())
print(f"{n_ok} of {feas.filter(like='predict').size} configurations can be graded")

**The objective:** the grade of §5, the scaled paths' R² against "no change", averaged over lead times
and equally over the stocks. One model per configuration learns all the stocks with the same settings.

**The optimizer:** Optuna's TPE sampler picks one step of each ladder per trial (it knows the steps'
order). The first quarter of the trials are spread at random; then it proposes more often near the
configurations that scored well. A proposal the data cannot support, or one already measured, is replaced
by its nearest unmeasured neighbour, so every trial adds a new configuration. Each runs §5 end to end; TEST
is not read until §9.

**Read with care:** configurations graded on fewer independent paths (coarse bars, long predictions) have
wider luck bands, and the best of many tries favours them; their scale β is also less certain, which costs
them R². The direction z and the bar for the best of all tries keep that honest. Ties at R² = 0 (scaled to
"no change") are ranked by direction z.

In [ ]:
res, study = ex.search_paths(panel, SP, kind=K["model"], grans=K["granularities"],
                             inputs=K["inputs"], outputs=K["outputs"], n_trials=K["n_trials"],
                             epochs=K["epochs"], draws=K["draws"], min_windows=K["min_windows"],
                             min_outcomes=K["min_outcomes"])
display(ex.results_table(res).head(12))

## 8 · How the three knobs change the picture

**The predicted paths themselves.** The decision of §5 under the best configuration of each bar size: the
same moment, very different questions (10 minutes to 3 sessions ahead) and very different answers. Solid:
the graded, scaled path; dotted: the network's own.

In [ ]:
best_g = ex.best_per(res, "granularity")  # the best of each bar size (ties: direction z)
runs = [(f"{int(r.granularity)}-min, input {int(r.input)}, predict {int(r.prediction)}",
         ex.run_path(panel, SP, (r.granularity, r.input, r.prediction), kind=K["model"],
                     epochs=K["epochs"], draws=K["draws"])) for r in best_g.itertuples()]
charts.path_picture(panel, ex.example_paths(panel, runs, NAME, TS), NAME, TS).show()

**Agreement by lead time.** Every configuration measured (grey) and the best of each bar size (colour): R²
against "no change", then the direction, at every point of the path against how many minutes ahead it is
(log scale). This is the clearest answer to "how far ahead is the path predictable?": where the coloured lines
sit above 0, and for how long.

In [ ]:
charts.lead_curves(res).show()
charts.lead_curves(res, metric="corr_h", title="Direction by lead time: the best "
                   "configuration of each bar size (VALIDATION)").show()

**Each knob on its own.** One panel per knob: every configuration (grey dots), the best (solid) and the
average (dotted) of each step. First the grade (R²; 0 = "no change"), then the direction's z against luck
(grey = where one configuration lands 95% of the time by luck, dashed = the bar the best of a step's tries
must clear).

In [ ]:
PARAMS = {"granularity": ("bar size (minutes)", K["granularities"]),
          "input": ("input length (bars)", K["inputs"]),
          "prediction": ("prediction length (bars)", K["outputs"])}
charts.param_effects(res, PARAMS).show()
charts.param_effects(res, PARAMS, metric="z").show()

**The maps.** One small map per bar size: input length up, prediction length across, coloured by the grade
on one shared scale (blue above 0, red below), each cell with its R² and its direction z. The best
configuration is framed; blank cells were not measured or cannot be.

In [ ]:
charts.granularity_maps(res, K["granularities"], K["inputs"], K["outputs"]).show()

**Which knob matters most,** and the optimizer's path trial by trial.

In [ ]:
imp = ex.importance(study)
if len(imp):
    charts.importance_bars(imp).show()
charts.search_history(res).show()
print(ex.search_words(res))

## 9 · TEST: does the winner hold on a month the optimizer never saw?

**Question: the optimizer chose the best of many configurations on VALIDATION. Is it still good on TEST?**

TEST is graded **once** here, for three configurations: the VALIDATION winner, the runner-up and the
worked example (`DEFAULT`, chosen before any result). Each is refit on TRAIN + VALIDATION (its held-out
sessions: the last ones of VALIDATION) and graded on TEST. A configuration **holds** if its TEST R² is
above 0 and its direction z is at least 1.65 (a 5% chance by luck). No TEST price, path or grade has been
shown before this cell (§2's data checks count TEST bars and their largest day move, not a direction).

In [ ]:
top2 = ex.best(res, k=2)
configs = [(lab, (int(r.granularity), int(r.input), int(r.prediction))) for lab, r in
           zip(("winner", "runner-up"), top2.itertuples(), strict=False)]
configs.append(("example", tuple(K["default"])))
tc = ex.test_paths(panel, SP, configs, kind=K["model"], epochs=K["epochs"], draws=K["draws"])
display(tc.set_index(["config", "part"])[["granularity", "input", "prediction", "r2", "beta",
                                           "r2_raw", "corr", "z", "share_pos", "ar1_r2"]].round(4))
charts.dots(ex.as_dots(tc, [f"{c} · {p.upper()}" for c, p in zip(tc["config"], tc["part"],
                                                                  strict=True)]),
            "corr", scale=1.0, unit="", title="Direction against luck: VALIDATION vs TEST").show()
print(ex.test_words(tc))

## 10 · The checks, the summary, what to try next

In [ ]:
table = rs.checks_table(CHECKS)
display(table)
win = ex.best(res).iloc[0]
held = tc[(tc["config"] == "winner") & (tc["part"] == "test")].iloc[0]
rs.scorecard({"notebook": "explorer", "smoke": K["smoke"], "model": K["model"],
              "names": NAMES, "window": SP["window"], "trials": len(res),
              "winner": {"granularity": win["granularity"], "input": win["input"],
                         "prediction": win["prediction"], "val_r2": win["r2"],
                         "val_z": win["z"], "beta": win["beta"], "test_r2": held["r2"],
                         "test_z": held["z"]},
              "above_no_change": int((res["r2"] > 0).sum()),
              "direction_above_luck": int((res["z"] >= 2).sum()),
              "checks_ok": f"{int(table['ok'].sum())}/{len(table)}"})

**Reading the result.** The summaries of §8 and §9 say it in words. Three outcomes are possible:

- **A configuration beats "no change" and holds on TEST**: that (bar size, input, prediction) is the
  baseline to build on.
- **A winner that does not hold**: the usual face of luck after many tries.
- **Nothing beats "no change"**: the networks' paths carry no information that survives a week. That is
  itself the baseline: for these stocks and months, the next path is best predicted as "no change", whatever
  the bar size and lengths.

**What to try next, one knob at a time:**

| next knob | how | why |
|---|---|---|
| the model | `MODEL` (§6) | TSMixer and DLinear read long inputs differently from TiDE |
| more months | `WINDOW` starting earlier (the 1-minute data begins 2025-06-18) | every luck band shrinks with more independent paths, most for coarse bars |
| more stocks | `SYMBOLS`, or `ex.PLAN` (more per group) | about 270 stocks pass the data rules |
| the target | a cross-sectional path (each stock minus the average of all) | removes the market, which no stock's own history predicts |
| the inputs | `strategy.build_inputs` groups (`use_daily`, `use_market`) | which inputs carry the information, if any |
| the last 6 months | `WINDOW = ("2026-04-01", "2026-09-30")`, `READ_LOCKED = True` | about 1,750 stocks with 1-minute data, at the price of the locked months |